# Source-forward comparison smoke test

This notebook compares the legacy event sampler with the optional source-forward annotation mode. The event-level prior should remain unchanged for the same random seed, while the source-forward run adds source stellar properties selected by the configured source magnitude cuts.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Find build/ whether Jupyter is launched from the repository root or tests/smoke/.
cwd = Path.cwd()
repo_root = cwd.parents[1] if cwd.match("*/tests/smoke") else cwd
build_dir = repo_root / "build"
if build_dir.exists():
    sys.path.insert(0, str(build_dir))

import genulens


## Simulation setup

The legacy run has no source-forward annotations. The source-forward run uses the same event seed, Roman photometry, and an apparent `F146mag` source-selection cut.


In [ ]:
N_SIMU = 2_000
SEED = 2026


def as_dataframe(result):
    return pd.DataFrame(result.to_numpy(), columns=result.columns)


def base_config():
    return genulens.Config(l=1.0, b=-3.9, n_simu=N_SIMU, seed=SEED)


cfg_legacy = base_config()
legacy = genulens.simulate(cfg_legacy)
df_legacy = as_dataframe(legacy)

cfg_forward = base_config()
cfg_forward.forward_source.enabled = 1
cfg_forward.forward_source.photometry = "roman"
cfg_forward.forward_source.min_initial_mass_msun = 0.1
cfg_forward.forward_source.max_initial_mass_msun = 1.0
cfg_forward.forward_source.match_source_selection = 1
cfg_forward.forward_source.selection_bands = ["F146mag"]
cfg_forward.forward_source.selection_min_magnitudes = [17.0]
cfg_forward.forward_source.selection_max_magnitudes = [24.0]

forward = genulens.simulate(cfg_forward)
df_forward = as_dataframe(forward)

assert len(df_legacy) == N_SIMU
assert len(df_forward) == N_SIMU

corner_columns = ["M_L", "D_L", "D_S", "mu_rel_N", "mu_rel_E"]
assert np.allclose(df_legacy[corner_columns], df_forward[corner_columns])

source_columns = ["M_S_ini", "M_S", "R_S", "teff_S", "logg_S", "theta_S", "M_F146mag_S"]
finite_source = np.isfinite(df_forward[source_columns].to_numpy()).all(axis=1)
assert finite_source.mean() > 0.95

apparent_f146 = df_forward["M_F146mag_S"] + 5.0 * np.log10(df_forward["D_S"]) - 5.0
selected = finite_source
assert np.all((apparent_f146[selected] >= 17.0) & (apparent_f146[selected] <= 24.0))

pd.DataFrame(
    {
        "legacy_rows": [len(df_legacy)],
        "forward_rows": [len(df_forward)],
        "finite_source_fraction": [finite_source.mean()],
    }
)


## Legacy vs source-forward event prior

This is the requested corner plot over `M_L`, `D_L`, `D_S`, `mu_rel_N`, and `mu_rel_E`. The same clipping used in the public example is applied before plotting.


In [ ]:
def event_corner_frame(df):
    mu_abs = np.sqrt(df["mu_rel_N"]**2 + df["mu_rel_E"]**2)
    return df[
        (mu_abs <= 20) &
        (df["M_L"] <= 1.5)
    ].copy()


df_legacy_corner = event_corner_frame(df_legacy)
df_forward_corner = event_corner_frame(df_forward)

try:
    import corner

    fig = corner.corner(
        df_legacy_corner[corner_columns].to_numpy(),
        labels=corner_columns,
        weights=df_legacy_corner["wtj"].to_numpy(),
        color="tab:blue",
        quantiles=[0.16, 0.50, 0.84],
        show_titles=True,
        title_fmt=".3g",
        label_kwargs={"fontsize": 10},
    )
    corner.corner(
        df_forward_corner[corner_columns].to_numpy(),
        labels=corner_columns,
        weights=df_forward_corner["wtj"].to_numpy(),
        color="tab:orange",
        fig=fig,
        show_titles=False,
    )
    fig.axes[0].plot([], [], color="tab:blue", label="legacy")
    fig.axes[0].plot([], [], color="tab:orange", label="source-forward")
    fig.axes[0].legend(frameon=False, loc="upper right")
except ImportError:
    fig, axes = plt.subplots(1, 2, figsize=(13, 6), constrained_layout=True)
    pd.plotting.scatter_matrix(
        df_legacy_corner[corner_columns],
        ax=axes[0],
        diagonal="hist",
        alpha=0.05,
    )
    pd.plotting.scatter_matrix(
        df_forward_corner[corner_columns],
        ax=axes[1],
        diagonal="hist",
        alpha=0.05,
    )
    axes[0].set_title("legacy")
    axes[1].set_title("source-forward")

fig.suptitle("Legacy vs source-forward event prior, |mu_rel| <= 20 & M_L <= 1.5", y=1.02)
plt.show()


## Source-forward full corner

This plot includes the event-level quantities plus source physical properties needed by the forward source-prior workflow.


In [ ]:
full_corner_columns = corner_columns + [
    "M_S_ini",
    "M_S",
    "R_S",
    "teff_S",
    "logg_S",
    "theta_S",
    "M_F146mag_S",
]

full_df = df_forward.loc[finite_source].copy()
full_df = event_corner_frame(full_df)

# Keep the smoke notebook responsive while preserving deterministic sampling.
if len(full_df) > 1_200:
    full_df = full_df.sample(n=1_200, random_state=SEED)

try:
    import corner

    fig = corner.corner(
        full_df[full_corner_columns].to_numpy(),
        labels=full_corner_columns,
        weights=full_df["wtj"].to_numpy(),
        quantiles=[0.16, 0.50, 0.84],
        show_titles=True,
        title_fmt=".3g",
        label_kwargs={"fontsize": 8},
    )
except ImportError:
    axes = pd.plotting.scatter_matrix(
        full_df[full_corner_columns],
        figsize=(14, 14),
        diagonal="hist",
        alpha=0.05,
    )
    fig = axes[0, 0].figure

fig.suptitle("Source-forward event and source-property prior", y=1.01)
plt.show()
